# Classify a video: d3-nano vs LFM3-VL-2B

Select a **Colab L4 or A100 GPU**, then run the cells in order and upload one short video. Both models use BF16. The notebook runs each model in a fresh Python process so its Transformers version is isolated and GPU memory is released afterward.

Labels: **Standing Still · Driving · Digging · Other**. Model scores are not calibrated accuracy probabilities; inspect the answers rather than declaring a winner from confidence alone. GPU execution has not been tested here.


In [ ]:
import sys, subprocess, json
from pathlib import Path
import torch
from google.colab import files
assert torch.cuda.is_available(), "Choose a GPU under Runtime > Change runtime type."
assert torch.cuda.get_device_capability()[0] >= 8, "Use L4/A100 or another native-BF16 GPU. T4 is not supported by this basic notebook."
print("GPU:", torch.cuda.get_device_name())
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one short video."
VIDEO = Path("/content/uploaded_video.mp4")
VIDEO.write_bytes(next(iter(uploaded.values())))
del uploaded
print("Video saved:", VIDEO)

In [ ]:
WORKER_SOURCE = "import sys,json,time\nimport cv2,numpy as np,torch\nfrom transformers import AutoModel,AutoProcessor,AutoModelForImageTextToText\nbackend,path,out=sys.argv[1:]\nlabels={\n 'standing_still':'Vehicle and working attachment stay still, ignoring minor vibration or foliage.',\n 'driving':'The vehicle itself travels, supported by changing viewpoint and background parallax.',\n 'digging':'The tool excavates, scrapes or scoops soil/material; equipment type alone is insufficient.',\n 'other':'Unclear, mixed, unsupported activity, including attachment movement without clear digging.'}\nstate='Silent chronological POV footage from construction equipment. Classify the dominant observable activity. Do not infer digging simply because an excavator is visible.'\nload_start=time.perf_counter()\nif backend=='d3':\n model=AutoModel.from_pretrained('vllm-sr/d3-nano',revision='8198c9f763b046605b2f4827b4999971fa2c92c4',trust_remote_code=True,device='cuda:0',batch_size=1)\n load_seconds=time.perf_counter()-load_start\n torch.cuda.synchronize();start=time.perf_counter()\n result=model.system_one(state=state,videos=[path],questions={'activity':{'type':'choice','instructions':'Which activity best describes this clip?','criteria':labels}})\n answer=result['answers']['activity']\n if 'probabilities' not in answer:raise RuntimeError(str(answer))\n scores=answer['probabilities']\nelse:\n revision='71d8e745d97c224984e0f20412f532a5ce9dfd3e'\n processor=AutoProcessor.from_pretrained('LiquidAI/LFM3-VL-2B',revision=revision,trust_remote_code=True)\n model=AutoModelForImageTextToText.from_pretrained('LiquidAI/LFM3-VL-2B',revision=revision,trust_remote_code=True,dtype=torch.bfloat16,device_map={'':'cuda:0'},attn_implementation='sdpa').eval()\n load_seconds=time.perf_counter()-load_start\n torch.cuda.synchronize();start=time.perf_counter()\n cap=cv2.VideoCapture(path);fps=cap.get(cv2.CAP_PROP_FPS);total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n if fps<=0 or total<4:raise ValueError('Invalid or too-short video')\n indices=np.linspace(0,total-1,min(max(int(total/fps*2),4),32,total)).round().astype(int).tolist()\n wanted=set(indices);frames=[];i=0\n while True:\n  ok,frame=cap.read()\n  if not ok:break\n  if i in wanted:\n   h,w=frame.shape[:2];scale=min(1,512/max(h,w))\n   frame=cv2.resize(frame,(round(w*scale),round(h*scale)))\n   frames.append(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))\n  i+=1\n cap.release()\n if len(frames)!=len(indices):raise ValueError('Incomplete video decoding')\n if len(frames)%2:frames.append(frames[-1]);indices.append(indices[-1])\n letters=list('ABCD');keys=list(labels)\n prompt=state+'\\nChoose one option:\\n'+'\\n'.join(f'{c}. {key}: {labels[key]}' for c,key in zip(letters,keys))+'\\nAnswer with exactly one letter A, B, C, or D.'\n messages=[{'role':'user','content':[{'type':'video'},{'type':'text','text':prompt}]}]\n text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)\n metadata={'fps':fps,'total_num_frames':total,'duration':total/fps,'frames_indices':indices,'width':frames[0].shape[1],'height':frames[0].shape[0]}\n inputs=processor(text=[text],videos=[np.stack(frames)],video_metadata=[metadata],do_sample_frames=False,return_tensors='pt')\n inputs={k:v.to('cuda:0') if hasattr(v,'to') else v for k,v in inputs.items() if k!='video_metadata'}\n ids=[processor.tokenizer.encode(c,add_special_tokens=False) for c in letters]\n if any(len(x)!=1 for x in ids):raise ValueError('Expected single-token letter options')\n with torch.inference_mode():\n  logits=model(**inputs,use_cache=False,logits_to_keep=1).logits[0,-1]\n  probabilities=torch.softmax(logits[[x[0] for x in ids]].float(),dim=-1).cpu().tolist()\n scores=dict(zip(keys,probabilities))\ntorch.cuda.synchronize()\nreport={'model':backend,'label':max(scores,key=scores.get),'option_scores':scores,'load_seconds':load_seconds,'classification_seconds':time.perf_counter()-start,'gpu':torch.cuda.get_device_name(),'torch':torch.__version__,'note':'Option scores are not calibrated accuracy probabilities.'}\nwith open(out,'w') as f:json.dump(report,f,indent=2)\nprint(json.dumps(report,indent=2))\n"
WORKER = Path("/content/classify_video.py")
WORKER.write_text(WORKER_SOURCE)

## Run d3-nano
Native decision API. The first run downloads the checkpoint.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.17.0", "accelerate", "safetensors", "einops", "opencv-python-headless==4.11.0.86"], check=True)
D3_RESULT = Path("/content/d3_result.json")
D3_RESULT.unlink(missing_ok=True)
subprocess.run([sys.executable, str(WORKER), "d3", str(VIDEO), str(D3_RESULT)], check=True)

## Run LFM3-VL-2B
Thinking is disabled. One forward pass scores the four allowed answer letters.


In [ ]:
# Transformers is imported only inside the workers, so changing its version
# does not require restarting this notebook kernel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.21.0"], check=True)
LFM_RESULT = Path("/content/lfm_result.json")
LFM_RESULT.unlink(missing_ok=True)
subprocess.run([sys.executable, str(WORKER), "lfm", str(VIDEO), str(LFM_RESULT)], check=True)

## Compare and download
Classification times include video preprocessing and the first model call; downloads/loading are separate. This is a sanity check, not an accuracy or optimized throughput benchmark.


In [ ]:
import matplotlib.pyplot as plt
results = [json.loads(D3_RESULT.read_text()), json.loads(LFM_RESULT.read_text())]
for result in results:
    print(f"{result['model']}: {result['label']} — {result['classification_seconds']:.2f}s")
labels = list(results[0]["option_scores"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, result in zip(axes, results):
    ax.bar(range(len(labels)), [result["option_scores"][k] for k in labels])
    ax.set_xticks(range(len(labels)), [k.replace("_", " ") for k in labels], rotation=25, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title(f"{result['model']}: {result['label']}")
axes[0].set_ylabel("Score within allowed options")
plt.tight_layout()
plt.show()
output = Path("/content/model_comparison.json")
output.write_text(json.dumps(results, indent=2))
files.download(str(output))